In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nfl_data_py as nfl

In [2]:
def pivot_wide_both_sides(data):
    import warnings
    warnings.filterwarnings('ignore')

    # Filter for valid play types
    data = data[data['play_type'].isin(['run', 'pass'])].copy()

    # Sort for consistency
    data = data.sort_values(['game_id', 'play_id'])

    # Helper function to expand player columns
    def expand_player_col(df, col_name):
        side = col_name.split('_')[0]  # offense or defense

        # Fill missing with empty string to allow split
        players_split = df[col_name].fillna('').str.split(';')

        # Pad each list to exactly 11 players (or truncate)
        players_list = players_split.apply(lambda lst: (lst + [None]*11)[:11])

        # Convert to DataFrame
        players_df = pd.DataFrame(players_list.tolist(), columns=[f'{side}_player_{i+1}' for i in range(11)])

        return players_df

    # Expand offense and defense player columns
    offense_wide = expand_player_col(data, 'offense_players')
    defense_wide = expand_player_col(data, 'defense_players')

    # Concatenate expanded columns with the original DataFrame
    pbp_wide = pd.concat([data.reset_index(drop=True), offense_wide, defense_wide], axis=1)

    # Drop potential duplicates
    pbp_wide = pbp_wide.drop_duplicates(subset=['game_id', 'play_id'])

    # Drop rows with NaN in offense_players or defense_players
    pbp_wide = pbp_wide.dropna(subset=['offense_players', 'defense_players'])

    return pbp_wide

In [3]:
def clean_depth_charts(depth_charts):
    # Remove duplicates on season, club_code, week, gsis_id
    depth_charts_copy = depth_charts.drop_duplicates(subset=['season', 'club_code', 'week', 'gsis_id'])
    depth_charts_copy = depth_charts_copy[depth_charts_copy['formation'].str.contains('Offense|Defense', regex=True)].copy()

    # Replace 'T' and 'G' in 'position' with the value from 'depth_position'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].isin(['T', 'G']),
        depth_charts_copy['depth_position'],
        depth_charts_copy['position']
    )

    # Replace 'LOT' in 'position'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].isin(['LOT']),
        'LT',
        depth_charts_copy['position']
    )

    # Replace 'ROT' in 'position'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].isin(['ROT']),
        'RT',
        depth_charts_copy['position']
    )

    # Replace 'RDE' and 'LDE'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].isin(['RDE', 'LDE']),
        'DE',
        depth_charts_copy['position']
    )   

    # Replace 'RDT' and 'LDT'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].isin(['RDT', 'LDT']),
        'DT',
        depth_charts_copy['position']
    )   
    
    # Replace 'DB' with 'CB'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].isin(['LB']),
        'MLB',
        depth_charts_copy['position']
    )

    # Replace 'K', 'P', 'LS'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].isin(["\n ", 'K', 'P', 'LS']),
        'special',
        depth_charts_copy['position']
    )

    # Replace 'DB' with 'CB'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].isin(['DB']),
        'CB',
        depth_charts_copy['position']
    )

    # Replace '\n'
    depth_charts_copy['position'] = np.where(
        depth_charts_copy['position'].str.contains(' '),
        'special',
        depth_charts_copy['position']
    )

    # Replace 'SBBYE'
    depth_charts_copy['week'] = np.where(
        depth_charts_copy['week'].isna(),
        '23',
        depth_charts_copy['week']
    )

    # Seniority counter
    depth_charts_copy['seniority'] = depth_charts_copy.groupby(['gsis_id']).cumcount() + 1

    # Cumulative sum of depth_team per gsis_team
    depth_charts_copy['depth_team'] = depth_charts_copy['depth_team'].astype(int)
    depth_charts_copy['depth_team_sum'] = depth_charts_copy.groupby(['gsis_id'])['depth_team'].cumsum()

    # Cumulative mean of depth_team per gsis_team
    depth_charts_copy['depth_team_mean'] = depth_charts_copy['depth_team_sum'] / depth_charts_copy['seniority']

    # Sort by season, club_code, week, position, depth_team_mean
    depth_charts_copy = depth_charts_copy.sort_values(['depth_team_mean'], ascending=True)

    # Cumulative count of season, week, club_team, position
    depth_charts_copy['depth_team_estimator'] = depth_charts_copy.groupby(['season', 'week', 'club_code', 'position']).cumcount() + 1
    
    #depth_charts_copy['depth_team_estimator'] = depth_charts_copy['depth_team_estimator'].astype(int)
    depth_charts_copy['depth_team_estimator'] = depth_charts_copy['depth_team_estimator'].astype(str)

    # Define position_label
    depth_charts_copy['position_label'] = depth_charts_copy['position'] + depth_charts_copy['depth_team_estimator'].astype(str)
    depth_charts_copy['season'] = depth_charts_copy['season'].astype(int)
    depth_charts_copy['gsis_id'] = depth_charts_copy['gsis_id'].astype(str)
    depth_charts_copy['week'] = depth_charts_copy['week'].astype(str).str.replace('.0', '', regex=False).astype(int)

    return depth_charts_copy

In [4]:
def pivot_long_both_sides(pbp_wide, depth_charts):
    import warnings
    warnings.filterwarnings('ignore')
    
    # Clean depth charts
    depth_charts_clean = clean_depth_charts(depth_charts)
    
    # Combine offense and defense player columns into long format
    long_df = pd.wide_to_long(
        pbp_wide.reset_index(),
        stubnames=['offense_player', 'defense_player'],
        i=['index'],  # preserves play-wise indexing
        j='player_number',
        sep='_',
        suffix='\\d+'
    ).reset_index()
    
    # Stack both sides
    offense_df = long_df[['game_id', 'play_id', 'offense_player', 'player_number', 'season', 'week']].copy()
    offense_df.rename(columns={'offense_player': 'player_id'}, inplace=True)

    defense_df = long_df[['game_id', 'play_id', 'defense_player', 'player_number', 'season', 'week']].copy()
    defense_df.rename(columns={'defense_player': 'player_id'}, inplace=True)
    
    all_long = pd.concat([offense_df, defense_df], ignore_index=True)

    # Deduplicate depth_charts BEFORE merge to prevent cartesian product
    depth_charts_dedup = depth_charts_clean.drop_duplicates(
        subset=['season', 'week', 'gsis_id']
    )[['season', 'week', 'gsis_id', 'position_label', 'full_name']]

    # Merge with depth charts
    all_long['player_id'] = all_long['player_id'].astype(str).str.strip()
    depth_charts_dedup['gsis_id'] = depth_charts_dedup['gsis_id'].astype(str).str.strip()
    merged = all_long.merge(
        depth_charts_dedup,
        left_on=['season', 'week', 'player_id'],
        right_on=['season', 'week', 'gsis_id'],
        how='left'
    )

    # Drop unnecessary
    merged = merged.drop(columns=['gsis_id'])

    # Replace the dropna with a fillna for debugging
    merged['position_label'] = merged['position_label'].fillna('special')

    # Filter nulls
    merged = merged.dropna(subset=['player_id', 'position_label'])

    # Deduplicate on the final result
    merged = merged.drop_duplicates(subset=['game_id', 'play_id', 'player_id'])

    return merged.sort_values(['game_id', 'play_id'])

In [5]:
def get_position_groups(_):
    """
    Returns a dictionary with keys for 'offense', 'defense', and position groups,
    and values as lists of hard-coded position_labels.
    Ignores the input and uses a fixed list.
    """
    position_labels = [
        'C1', 'C2', 'C3', 'C4', 'CB1', 'CB2', 'CB3', 'CB4', 'CB5', 'CB6', 'CB7', 'CB8',
        'DE1', 'DE2', 'DE3', 'DE4', 'DE5', 'DE6', 'DE7', 'DT1', 'DT2', 'DT3', 'DT4', 'DT5', 'DT6', 'DT7',
        'FB1', 'FB2', 'FS1', 'FS2', 'FS3', 'FS4', 'ILB1', 'ILB2', 'ILB3', 'ILB4', 'ILB5', 'ILB6',
        'LG1', 'LG2', 'LG3', 'LT1', 'LT2', 'LT3', 'MLB1', 'MLB2', 'MLB3', 'NT1', 'NT2', 'NT3',
        'OLB1', 'OLB2', 'OLB3', 'OLB4', 'OLB5', 'OLB6', 'QB1', 'QB2', 'QB3', 'RB1', 'RB2', 'RB3', 'RB4', 'RB5',
        'RG1', 'RG2', 'RG3', 'RT1', 'RT2', 'RT3', 'SS1', 'SS2', 'SS3', 'SS4', 'SS5',
        'TE1', 'TE2', 'TE3', 'TE4', 'TE5', 'WR1', 'WR2', 'WR3', 'WR4', 'WR5', 'WR6', 'WR7', 'special1'
    ]

    group_dict = {
        'offense': [p for p in position_labels if any(p.startswith(x) for x in [
            'QB', 'RB', 'FB', 'WR', 'TE', 'LT', 'LG', 'C', 'RG', 'RT'])],
        'defense': [p for p in position_labels if any(p.startswith(x) for x in [
            'DE', 'DT', 'NT', 'OLB', 'ILB', 'MLB', 'CB', 'FS', 'SS'])],
        'qbs': [p for p in position_labels if p.startswith('QB')],
        'rbs': [p for p in position_labels if p.startswith('RB') or p.startswith('FB')],
        'tes': [p for p in position_labels if p.startswith('TE')],
        'wrs': [p for p in position_labels if p.startswith('WR')],
        'ol': [p for p in position_labels if p.startswith(('LT', 'LG', 'C', 'RG', 'RT')) and not p.startswith('CB')],
        'dl': [p for p in position_labels if p.startswith(('DE', 'DT', 'NT'))],
        'lbs': [p for p in position_labels if p.startswith(('OLB', 'ILB', 'MLB'))],
        'cbs': [p for p in position_labels if p.startswith('CB')],
        'ss': [p for p in position_labels if p.startswith(('SS', 'FS'))],
        'special': [p for p in position_labels if p.startswith('special')]
    }

    return group_dict

position_groups = get_position_groups(None)

In [6]:
tackling_columns = ['game_id', 'play_id', 
    'play_type', 
    'complete_pass',
    'passer_player_id',
    'receiver_player_id',
    'rusher_player_id',
    'punt_returner_player_id',
    'kickoff_returner_player_id',
     'assist_tackle_1_player_id', 
    'assist_tackle_2_player_id', 
     'assist_tackle_3_player_id', 
     'assist_tackle_4_player_id',
     'solo_tackle_1_player_id', 
    'solo_tackle_2_player_id']

In [7]:
rare_play_type_cols = ['play_type_nan',
 'offense_personnel_','offense_personnel_0 RB, 0 TE, 0 WR,1 P,1 LS,3 DL,1 K','offense_personnel_0 RB, 0 TE, 0 WR,1 P,4 LB,1 LS,5 DB','offense_personnel_0 RB, 0 TE, 1 WR,1 P,3 LB,1 LS,5 DB','offense_personnel_0 RB, 0 TE, 2 WR,1 P,5 LB,1 LS,2 DB','offense_personnel_0 RB, 0 TE, 5 WR',
 'offense_personnel_0 RB, 1 TE, 0 WR,1 P,1 LS,2 DL,1 K','offense_personnel_0 RB, 1 TE, 0 WR,1 P,3 LB,1 LS,1 DL,4 DB','offense_personnel_0 RB, 1 TE, 0 WR,1 P,4 LB,1 LS,4 DB','offense_personnel_0 RB, 1 TE, 0 WR,1 P,5 LB,1 LS,3 DB',
 'offense_personnel_0 RB, 1 TE, 1 WR,1 P,3 LB,1 LS,1 DL,3 DB','offense_personnel_0 RB, 1 TE, 1 WR,1 P,3 LB,1 LS,4 DB','offense_personnel_0 RB, 1 TE, 1 WR,1 P,4 LB,1 LS,1 DL,2 DB','offense_personnel_0 RB, 1 TE, 1 WR,1 P,4 LB,1 LS,3 DB',
 'offense_personnel_0 RB, 1 TE, 1 WR,1 P,5 LB,1 LS,2 DB','offense_personnel_0 RB, 1 TE, 2 WR,1 P,3 LB,1 LS,3 DB','offense_personnel_0 RB, 1 TE, 3 WR,1 DB','offense_personnel_0 RB, 2 TE, 0 WR,1 P,1 LS,1 DL,1 K',
 'offense_personnel_0 RB, 2 TE, 0 WR,1 P,1 LS,2 DL,1 K','offense_personnel_0 RB, 2 TE, 0 WR,1 P,3 LB,1 LS,4 DB','offense_personnel_0 RB, 2 TE, 2 WR,1 DB','offense_personnel_0 RB, 2 TE, 2 WR,1 P,3 LB,1 LS,2 DB',
 'offense_personnel_0 RB, 2 TE, 3 WR','offense_personnel_0 RB, 3 TE, 2 WR','offense_personnel_0 RB, 4 TE, 1 WR','offense_personnel_1 RB, 0 TE, 0 WR,1 P,3 LB,1 LS,1 DL,4 DB',
 'offense_personnel_1 RB, 0 TE, 0 WR,1 P,4 LB,1 LS,3 DB','offense_personnel_1 RB, 0 TE, 1 WR,1 P,3 LB,1 LS,1 DL,3 DB','offense_personnel_1 RB, 0 TE, 1 WR,1 P,3 LB,1 LS,4 DB','offense_personnel_1 RB, 0 TE, 1 WR,1 P,4 LB,1 LS,3 DB',
 'offense_personnel_1 RB, 0 TE, 2 WR,1 P,4 LB,1 LS,2 DB','offense_personnel_1 RB, 0 TE, 3 WR,1 DB','offense_personnel_1 RB, 0 TE, 3 WR,1 DL','offense_personnel_1 RB, 1 TE, 0 WR,1 P,3 LB,1 LS,1 DL,3 DB',
 'offense_personnel_1 RB, 1 TE, 0 WR,1 P,3 LB,1 LS,4 DB','offense_personnel_1 RB, 1 TE, 0 WR,1 P,4 LB,1 LS,3 DB','offense_personnel_1 RB, 1 TE, 1 WR,1 P,2 LB,1 LS,1 DL,3 DB','offense_personnel_1 RB, 1 TE, 1 WR,1 P,3 LB,1 LS,1 DL,2 DB',
 'offense_personnel_1 RB, 1 TE, 1 WR,1 P,3 LB,1 LS,2 DB','offense_personnel_1 RB, 1 TE, 1 WR,1 P,4 LB,1 LS,1 DL,1 DB','offense_personnel_1 RB, 1 TE, 1 WR,1 P,4 LB,1 LS,2 DB','offense_personnel_1 RB, 1 TE, 2 WR',
 'offense_personnel_1 RB, 1 TE, 2 WR,1 DB','offense_personnel_1 RB, 1 TE, 2 WR,1 DL','offense_personnel_1 RB, 1 TE, 2 WR,1 LB','offense_personnel_1 RB, 1 TE, 2 WR,1 P,3 LB,1 LS,2 DB',
 'offense_personnel_1 RB, 1 TE, 3 WR,1 P,3 LB,1 LS,1 DB','offense_personnel_1 RB, 1 TE, 4 WR','offense_personnel_1 RB, 2 TE, 0 WR,1 P,2 LB,1 LS,1 DL,3 DB','offense_personnel_1 RB, 2 TE, 0 WR,2 DL',
 'offense_personnel_1 RB, 2 TE, 1 WR,1 DB','offense_personnel_1 RB, 2 TE, 1 WR,1 DL','offense_personnel_1 RB, 2 TE, 1 WR,1 LB','offense_personnel_1 RB, 2 TE, 1 WR,1 P,2 LB,1 LS,3 DB',
 'offense_personnel_1 RB, 2 TE, 1 WR,1 P,3 LB,1 LS,2 DB','offense_personnel_1 RB, 2 TE, 1 WR,1 P,4 LB,1 LS,1 DB','offense_personnel_1 RB, 2 TE, 3 WR','offense_personnel_1 RB, 3 TE, 0 WR',
 'offense_personnel_1 RB, 3 TE, 0 WR,1 DB','offense_personnel_1 RB, 3 TE, 0 WR,1 DL','offense_personnel_1 RB, 3 TE, 0 WR,1 LB','offense_personnel_1 RB, 4 TE, 0 WR','offense_personnel_2 QB, 0 RB, 1 TE, 3 WR',
 'offense_personnel_2 QB, 0 RB, 2 TE, 1 WR,1 DL','offense_personnel_2 QB, 1 RB, 0 TE, 3 WR','offense_personnel_2 QB, 1 RB, 1 TE, 1 WR,1 DL','offense_personnel_2 QB, 1 RB, 2 TE, 0 WR,1 DL',
 'offense_personnel_2 QB, 1 RB, 2 TE, 1 WR', 'offense_personnel_2 QB, 1 RB, 3 TE, 0 WR','offense_personnel_2 QB, 2 RB, 0 TE, 2 WR','offense_personnel_2 QB, 2 RB, 1 TE, 1 WR','offense_personnel_2 QB, 2 RB, 2 TE, 0 WR',
 'offense_personnel_2 QB, 3 RB, 1 TE, 0 WR','offense_personnel_2 QB, 6 OL, 1 RB, 1 TE, 0 WR,1 DL','offense_personnel_2 QB, 6 OL, 1 RB, 1 TE, 1 WR','offense_personnel_2 QB, 6 OL, 1 RB, 2 TE, 0 WR',
 'offense_personnel_2 QB, 6 OL, 2 RB, 0 TE, 1 WR','offense_personnel_2 RB, 0 TE, 1 WR,1 P,1 LS,1 DB','offense_personnel_2 RB, 0 TE, 1 WR,1 P,4 LB,1 LS,2 DB','offense_personnel_2 RB, 0 TE, 2 WR,1 P,3 LB,1 LS,2 DB',
 'offense_personnel_2 RB, 0 TE, 2 WR,1 P,4 LB,1 LS,1 DB','offense_personnel_2 RB, 1 TE, 0 WR,1 P,2 LB,1 LS,4 DB','offense_personnel_2 RB, 1 TE, 0 WR,1 P,3 LB,1 LS,2 DB','offense_personnel_2 RB, 1 TE, 0 WR,1 P,4 LB,1 LS,2 DB',
 'offense_personnel_2 RB, 1 TE, 1 WR,1 DB','offense_personnel_2 RB, 1 TE, 1 WR,1 P,2 LB,1 LS,1 DL,2 DB','offense_personnel_2 RB, 1 TE, 1 WR,1 P,3 LB,1 LS,2 DB','offense_personnel_2 RB, 1 TE, 1 WR,1 P,4 LB,1 LS,1 DB',
 'offense_personnel_2 RB, 1 TE, 2 WR,1 P,2 LB,1 LS,2 DB','offense_personnel_2 RB, 1 TE, 3 WR','offense_personnel_2 RB, 2 TE, 0 WR,1 DB','offense_personnel_2 RB, 2 TE, 0 WR,1 DL',
 'offense_personnel_2 RB, 2 TE, 0 WR,1 P,2 LB,1 LS,3 DB','offense_personnel_2 RB, 2 TE, 0 WR,1 P,3 LB,1 LS,2 DB','offense_personnel_2 RB, 2 TE, 1 WR,1 P,2 LB,1 LS,2 DB','offense_personnel_2 RB, 3 TE, 0 WR',
 'offense_personnel_2 RB, 3 TE, 1 WR','offense_personnel_2 RB, 3 TE, 2 WR','offense_personnel_3 QB, 1 RB, 1 TE, 1 WR','offense_personnel_3 RB, 0 TE, 0 WR,1 P,4 LB,1 LS,2 DB',
 'offense_personnel_3 RB, 0 TE, 1 WR,1 P,2 LB,1 LS,1 DL,2 DB','offense_personnel_3 RB, 0 TE, 1 WR,1 P,4 LB,1 LS,1 DB','offense_personnel_3 RB, 0 TE, 2 WR','offense_personnel_3 RB, 1 TE, 1 WR',
 'offense_personnel_3 RB, 1 TE, 1 WR,1 P,3 LB,1 LS,1 DB','offense_personnel_3 RB, 1 TE, 3 WR','offense_personnel_3 RB, 2 TE, 0 WR','offense_personnel_3 RB, 3 TE, 3 WR',
 'offense_personnel_4 RB, 1 TE, 0 WR','offense_personnel_6 OL, 0 RB, 0 TE, 0 WR,1 P,1 LS,2 DL,1 K','offense_personnel_6 OL, 0 RB, 0 TE, 0 WR,4 DL','offense_personnel_6 OL, 0 RB, 0 TE, 3 WR,1 DL',
 'offense_personnel_6 OL, 0 RB, 1 TE, 0 WR,1 P,1 LS,1 DL,1 K',
 'offense_personnel_6 OL, 0 RB, 1 TE, 3 WR','offense_personnel_6 OL, 0 RB, 2 TE, 0 WR,1 LS,1 K','offense_personnel_6 OL, 0 RB, 2 TE, 0 WR,1 P,1 LS,1 K','offense_personnel_6 OL, 1 RB, 0 TE, 2 WR,1 LB',
 'offense_personnel_6 OL, 1 RB, 1 TE, 0 WR,1 LB,1 DB','offense_personnel_6 OL, 1 RB, 1 TE, 0 WR,1 P,1 LS,1 K','offense_personnel_6 OL, 1 RB, 1 TE, 0 WR,2 DL','offense_personnel_6 OL, 1 RB, 1 TE, 1 WR,1 DL',
 'offense_personnel_6 OL, 1 RB, 1 TE, 1 WR,1 LB','offense_personnel_6 OL, 1 RB, 2 TE, 0 WR,1 DL','offense_personnel_6 OL, 1 RB, 2 TE, 0 WR,1 LB','offense_personnel_6 OL, 1 RB, 3 TE, 0 WR',
 'offense_personnel_6 OL, 2 RB, 0 TE, 2 WR','offense_personnel_6 OL, 2 RB, 1 TE, 0 WR,1 DL','offense_personnel_6 OL, 3 RB, 0 TE, 1 WR','offense_personnel_7 OL, 0 RB, 0 TE, 0 WR,1 P,1 LS,1 DL,1 K',
 'offense_personnel_7 OL, 0 RB, 1 TE, 0 WR,1 P,1 DL,1 K','offense_personnel_7 OL, 0 RB, 1 TE, 0 WR,1 P,1 LS,1 K','offense_personnel_7 OL, 0 RB, 2 TE, 0 WR,1 P','offense_personnel_7 OL, 1 RB, 0 TE, 2 WR','offense_personnel_7 OL, 1 RB, 1 TE, 0 WR,1 LB',
 'offense_personnel_7 OL, 1 RB, 1 TE, 1 WR','offense_personnel_7 OL, 1 RB, 2 TE, 0 WR','offense_personnel_7 OL, 2 RB, 0 TE, 1 WR','offense_personnel_7 OL, 2 RB, 1 TE, 0 WR','offense_personnel_7 OL, 2 RB, 2 TE, 0 WR','offense_personnel_nan']

In [8]:
rare_cols = ['primary_injury_Abdomen','primary_injury_Achilles','primary_injury_Ankles','primary_injury_Biceps','primary_injury_Chest','primary_injury_Collarbone','primary_injury_Core','primary_injury_Core Muscle','primary_injury_Core Muscle Injury',
 'primary_injury_Elbow','primary_injury_Eye','primary_injury_Fibula','primary_injury_Finger','primary_injury_Forearm','primary_injury_Hand','primary_injury_Head','primary_injury_Heel','primary_injury_Hip Flexor','primary_injury_Hips','primary_injury_Jaw',
 'primary_injury_Knees','primary_injury_Lower Leg','primary_injury_Lower leg','primary_injury_Lumbar','primary_injury_Medical Illness','primary_injury_Mouth','primary_injury_Nose','primary_injury_Not Injury Related','primary_injury_Oblique',
 'primary_injury_Pectoral','primary_injury_Quadricep','primary_injury_Rib','primary_injury_Ribs','primary_injury_Right Collarbone','primary_injury_Shin','primary_injury_Sternoclavicular','primary_injury_Stinger','primary_injury_Teeth',
 'primary_injury_Thigh','primary_injury_Thumb','primary_injury_Tibia','primary_injury_Toe','primary_injury_Trapezius','primary_injury_Tricep','primary_injury_Triceps','primary_injury_Wrist','primary_injury_left Ankle','primary_injury_left Knee',
 'primary_injury_left Shoulder','primary_injury_left Wrist','primary_injury_right Elbow','primary_injury_right Foot','primary_injury_right Groin','primary_injury_right Hamstring','primary_injury_right Hand','primary_injury_right Hip','primary_injury_right Knee',
 'primary_injury_right Quadricep','primary_injury_right Shoulder','primary_injury_right Thigh','primary_injury_right Thumb','primary_injury_right Wrist','primary_injury_right hip','secondary_injury_Achilles','secondary_injury_Ankle',
 'secondary_injury_Ankle / Shoulder','secondary_injury_Ankle, Ankle','secondary_injury_Ankle, Hamstring','secondary_injury_Ankles','secondary_injury_Back','secondary_injury_Back, Ankle','secondary_injury_Biceps','secondary_injury_Calf','secondary_injury_Chest',
 'secondary_injury_Concussion','secondary_injury_Core','secondary_injury_Elbow','secondary_injury_Finger','secondary_injury_Foot','secondary_injury_Groin','secondary_injury_Groin, Wrist','secondary_injury_Hamstring','secondary_injury_Hamstring, Wrist',
 'secondary_injury_Hand','secondary_injury_Hand, Groin','secondary_injury_Heel','secondary_injury_Hip','secondary_injury_Illness','secondary_injury_Knee','secondary_injury_Knee Illness','secondary_injury_Knee, Achilles','secondary_injury_Neck',
 'secondary_injury_Not Injury Related','secondary_injury_Oblique','secondary_injury_Pelvis','secondary_injury_Personal','secondary_injury_Quadricep','secondary_injury_Quadricep, Ankle, Knee','secondary_injury_Rib','secondary_injury_Rib, Illness',
 'secondary_injury_Ribs','secondary_injury_Ribs Knee','secondary_injury_Ribs, Back','secondary_injury_Shoulder','secondary_injury_Shoulder, Knee','secondary_injury_Sternum','secondary_injury_Stinger','secondary_injury_Thigh',
 'secondary_injury_Thumb','secondary_injury_Toe','secondary_injury_Toe, Achilles, Hip','secondary_injury_Toe, Hamstring','secondary_injury_Tooth','secondary_injury_Triceps','secondary_injury_Wrist','secondary_injury_shoulder, illness']

In [9]:
def data_preparation(pbp_data, depth_charts, weekly_rosters, context_forecast_df, injuries, prev_year):
    # Define the ball carrier
    pbp_data['ball_carrier'] = np.where(
        (pbp_data['play_type'] == 'pass') & (pbp_data['complete_pass'] == True),
        pbp_data['receiver_player_id'],
        np.where(
            (pbp_data['play_type'] == 'pass') & (pbp_data['complete_pass'] == False),
            pbp_data['passer_player_id'],
            np.where(
                pbp_data['play_type'] == 'run',
                pbp_data['rusher_player_id'],
                np.where(
                    pbp_data['play_type'] == 'punt',
                    pbp_data['punt_returner_player_id'],
                    np.where(
                        pbp_data['play_type'] == 'kickoff',
                        pbp_data['kickoff_returner_player_id'],
                        None
                    )
                )
            )
        )
    )

    # Define tacklers
    pbp_data['tackler_ids'] = pbp_data[tackling_columns[5:]].apply(lambda row: row.dropna().tolist(), axis=1)
    pbp_data['tackler1'] = pbp_data['tackler_ids'].apply(lambda x: x[0] if len(x) > 0 else None)
    pbp_data['tackler2'] = pbp_data['tackler_ids'].apply(lambda x: x[1] if len(x) > 1 else None)

    # Define binary tackle indicator
    pbp_data['tackle_occurred'] = np.where(pbp_data['tackler1'].notna(), 1, 0)

    # Call pivot_wide and pivot_long function
    pbp_long = pivot_long_both_sides(pivot_wide_both_sides(pbp_data[['offense_players', 'defense_players', 'game_id', 'play_id', 'play_type', 'season', 'week']]), depth_charts)

    # Merge pbp_data with pbp_long to get base columns for feature engineering
    pbp_long = pbp_long.merge(pbp_data[['game_id', 'play_id', 'yards_gained', 'was_pressure', 'time_to_throw', 'yardline_100', 'xpass', 'air_yards', 'offense_personnel', 'run_location',
                                        'shotgun', 'no_huddle', 'qb_scramble', 'play_type']].copy(), 
                                on=['game_id', 'play_id'], how='left')

    # Merge with tackle columns
    pbp_long = pbp_long.merge(pbp_data[['game_id', 'play_id', 'ball_carrier', 'tackle_occurred']], left_on=['game_id', 'play_id', 'player_id'], right_on=['game_id', 'play_id', 'ball_carrier'], how='left')
    pbp_long = pbp_long.merge(pbp_data[['game_id', 'play_id',  'tackler1', 'tackle_occurred']], left_on=['game_id', 'play_id', 'player_id'], right_on=['game_id', 'play_id', 'tackler1'], how='left')
    pbp_long = pbp_long.merge(pbp_data[['game_id', 'play_id',  'tackler2', 'tackle_occurred']], left_on=['game_id', 'play_id', 'player_id'], right_on=['game_id', 'play_id', 'tackler2'], how='left')
    pbp_long['tackle_occurred'] = pbp_long[['tackle_occurred_x', 'tackle_occurred_y', 'tackle_occurred']].max(axis=1)
    pbp_long = pbp_long.drop(columns=['tackle_occurred_x', 'tackle_occurred_y'])

    # Convert play types to dummy variables
    play_type_dummies = pd.get_dummies(pbp_long[['play_type', 'offense_personnel', 'run_location']], prefix=['play_type', 'offense_personnel', 'run_location'], dummy_na=True)
    #print(play_type_dummies.shape)

    # Combine sparse dummy variables into 'rare_play_type'
    existing_rare = [c for c in rare_play_type_cols if c in play_type_dummies.columns]
    play_type_dummies['rare_play_type'] = play_type_dummies[existing_rare].any(axis=1).astype(int)
    play_type_dummies = play_type_dummies.drop(columns=existing_rare)
    #print(play_type_dummies.shape)
    pbp_long = pd.concat([pbp_long, play_type_dummies], axis=1)
    pbp_long = pbp_long.drop(columns=['play_type', 'offense_personnel', 'run_location'])

    # Merge with weekly_rosters to get height, weight, age
    pbp_long = pbp_long.merge(weekly_rosters[['season', 'week', 'player_id', 'height', 'weight', 'age']], 
                            on=['season', 'week', 'player_id'], how='left')

    # Merge residual_epa
    pbp_long = pbp_long.merge(context_forecast_df[['game_id', 'play_id', 'residual_epa']], on=['game_id', 'play_id'], how='left')

    pbp_long = pbp_long.drop(columns=['ball_carrier', 'tackler1', 'tackler2', 'player_number'])

    # Assuming pbp_long is your final pre-grouped DataFrame
    df = pbp_long.sort_values(['game_id', 'player_id', 'play_id']).copy()
    df['snaps'] = 1 # Each row represents a snap

    # 1. Define the grouping key
    group_cols = ['game_id', 'player_id']

    # 2. Identify columns for different aggregation types
    all_cols = set(df.columns)

    # Define columns that represent a state or identification *other than* the grouping key
    # We will use 'last' for these.
    state_cols = ['season', 'week', 'full_name', 'position_label', 'height', 'weight', 'age']

    # All remaining columns should be summed (play indicators and point totals)
    avg_cols = ['yards_gained', 'was_pressure', 'time_to_throw', 'yardline_100', 'xpass', 'air_yards']
    sum_cols = list(all_cols - set(group_cols) - set(state_cols) - set(avg_cols))

    # 3. Create the aggregation dictionary
    agg_dict = {col: 'sum' for col in sum_cols}
    agg_dict.update({col: 'mean' for col in avg_cols})
    agg_dict.update({col: 'last' for col in state_cols}) # Use 'last' for state columns that should reflect the final play's value (e.g., player size)

    # 4. Perform the aggregation
    # This is where the fix takes effect, as 'game_id' and 'player_id' are no longer keys in agg_dict.
    final_cumulative_df = df.groupby(group_cols).agg(agg_dict).reset_index()
    final_cumulative_df = final_cumulative_df.drop(columns=['play_id'])

    # Merge with injuries
    df = final_cumulative_df.merge(injuries[['season', 'week', 'gsis_id', 'report_primary_injury', 'report_secondary_injury', 'report_status']], 
                            left_on=['season', 'week', 'player_id'], right_on=['season', 'week', 'gsis_id'], how='left')

    # Identify records in injuries not present in df
    injuries['season'] = injuries['season'].astype(int)
    injuries['week'] = injuries['week'].astype(int)
    injuries['key'] = injuries['season'].astype(str) + '_' + injuries['week'].astype(str) + '_' + injuries['gsis_id'].astype(str)
    df['key'] = df['season'].astype(str) + '_' + df['week'].astype(str) + '_' + df['player_id'].astype(str)
    injury_not_in_pbp = injuries[~injuries['key'].isin(df['key'])]

    # Append missing injury records to df with NaNs for other columns
    missing_injury_records = injuries[injuries['key'].isin(injury_not_in_pbp['key'])]
    missing_injury_records = missing_injury_records[['season', 'week', 'gsis_id', 'report_primary_injury', 'report_secondary_injury', 'report_status']]
    missing_injury_records = missing_injury_records.rename(columns={'gsis_id': 'player_id'})
    for col in df.columns:
        if col not in missing_injury_records.columns:
            missing_injury_records[col] = np.nan
    df = pd.concat([df, missing_injury_records], ignore_index=True)
    df = df.drop(columns=['key', 'gsis_id'])

    # Convert injury columns to dummy variables
    injury_dummies = pd.get_dummies(df[['report_primary_injury', 'report_secondary_injury', 'report_status']], prefix=['primary_injury', 'secondary_injury', 'status'], dummy_na=True)
    print(injury_dummies.shape)

    # CRASH FIX: Only check for rare_cols that actually exist in THIS year's dummies
    existing_rare = [c for c in rare_cols if c in injury_dummies.columns]
    if existing_rare:
        injury_dummies['rare_injury'] = injury_dummies[existing_rare].any(axis=1).astype(int)
        injury_dummies = injury_dummies.drop(columns=existing_rare)
    else:
        injury_dummies['rare_injury'] = 0
    injury_dummies = injury_dummies.drop(columns=rare_cols, errors='ignore')
    print(injury_dummies.shape)
    df = pd.concat([df, injury_dummies], axis=1)
    df = df.drop(columns=['report_primary_injury', 'report_secondary_injury', 'report_status'])

    # --- NEW SECTION: INCORPORATE PREVIOUS YEAR SEED DATA ---
    # This is where we "prime" the state for the new season
    if prev_year is not None and not (isinstance(prev_year, list) and len(prev_year) == 0):
        prev_year['is_seed'] = True
        df['is_seed'] = False
        df = pd.concat([prev_year, df], ignore_index=True)
    else:
        df['is_seed'] = False

    # Sort by game_id and play_id
    df = df.sort_values(['season', 'week'])
    grouped = df.groupby('player_id')

    # Loop through to calculate rolling averages for each column
    idx_cols = ['game_id', 'player_id', 'season', 'week', 'position_label', 'full_name', 'is_seed']
    for col in df.columns:
        if col not in idx_cols and pd.api.types.is_numeric_dtype(df[col]):
            df[f'rolling_{col}'] = grouped[col].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

    # Cumulative stats
    df['cumulative_snaps'] = grouped['snaps'].cumsum()
    df['cumulative_tackles'] = grouped['tackle_occurred'].cumsum()
    df['cumulative_yards'] = grouped['yards_gained'].cumsum()
    df['cumulative_pressure'] = grouped['was_pressure'].cumsum()
    df['cumulative_time_to_throw'] = grouped['time_to_throw'].cumsum()
    df['cumulative_air_yards'] = grouped['air_yards'].cumsum()
    df['cumulative_xpass'] = grouped['xpass'].cumsum()
    df['cumulative_qb_scramble'] = grouped['qb_scramble'].cumsum()
    df['cumulative_no_huddle'] = grouped['no_huddle'].cumsum()
    df['cumulative_shotgun'] = grouped['shotgun'].cumsum()
    df['cumulative_yardline_100'] = grouped['yardline_100'].cumsum()

    status_cols = df.columns[df.columns.str.contains('status')]
    for col in status_cols:
        df[f'cumulative_{col}'] = grouped[col].cumsum()

    # A. Workload Spike Feature (Acute-to-Chronic Workload Ratio Proxy)
    # We assume the rolling window operates on plays, not games, given the original code structure.
    # Acute (Short-term) window = 4 games/plays, Chronic (Long-term) window = 16 games/plays

    short_window = 4
    long_window = 16

    df[f'rolling_sum_snaps_{short_window}'] = grouped['snaps'].transform(
        lambda x: x.rolling(window=short_window, min_periods=1).sum()
    )

    df[f'rolling_sum_snaps_{long_window}'] = grouped['snaps'].transform(
        lambda x: x.rolling(window=long_window, min_periods=1).sum()
    )

    # Load Spike: Ratio of short-term load (4 plays) to long-term load (16 plays)
    # Add a small constant (1) to the denominator to prevent division by zero
    df['snap_load_spike_ratio'] = df[f'rolling_sum_snaps_{short_window}'] / (df[f'rolling_sum_snaps_{long_window}'] + 1)


    # B. Performance Trend (Decay/Decline)
    # Rate of change in Expected Points Added (EPA) per snap over the last 3 plays/games.
    if 'cumulative_snaps' in df.columns and 'cumulative_epa' in df.columns:
        df['cumulative_epa_per_snap'] = df['cumulative_epa'] / (df['cumulative_snaps'].replace(0, np.nan))

        # Calculate the change (delta) in the rolling mean EPA/Snap
        df['rolling_mean_epa_per_snap_3'] = grouped['cumulative_epa_per_snap'].transform(
            lambda x: x.rolling(window=3, min_periods=1).mean()
        )
        
        # Lag the rolling mean by 1 play to find the change from the previous play's average
        df['rolling_mean_epa_per_snap_lag_1'] = grouped['rolling_mean_epa_per_snap_3'].shift(1)
        
        # Performance Delta: Current rolling average minus the previous rolling average
        df['rolling_epa_decline_delta'] = df['rolling_mean_epa_per_snap_3'] - df['rolling_mean_epa_per_snap_lag_1']
        
        df = df.drop(columns=['rolling_mean_epa_per_snap_lag_1']) # Cleanup temp column


    # C. Physical Load Interaction
    # Cumulative load weighted by player mass (assumes 'weight' is available)
    if 'weight' in df.columns:
        df['cumulative_snap_weight_load'] = df['cumulative_snaps'] * df['weight']


    # D. Weighted Injury History (EWMA)
    # Gives recent injury status exponential priority over older status.
    # We use the maximum injury status (O=4, D=3, Q=2, P=1, None=0)
    df['injury_status_risk'] = 0.0
    if 'status_Out' in df.columns:
        df['injury_status_risk'] += df['status_Out'] * 4 
    if 'status_Doubtful' in df.columns:
        df['injury_status_risk'] += df['status_Doubtful'] * 3
    if 'status_Questionable' in df.columns:
        df['injury_status_risk'] += df['status_Questionable'] * 2
    if 'status_Probable' in df.columns:
        df['injury_status_risk'] += df['status_Probable'] * 1

    # Calculate EWMA, common alpha is between 0.9 and 0.95 for recent history
    df['ewma_injury_risk'] = grouped['injury_status_risk'].transform(
        lambda x: x.ewm(span=14, adjust=False, min_periods=1).mean()
    )

    # E. Mass change
    # Shows change in recent weight measurements
    df['rolling_mean_weight_3'] = grouped['weight'].transform(
        lambda x: x.rolling(window=long_window, min_periods=1).mean()
    )
    df['weight_change'] = df['weight'] - df['rolling_mean_weight_3']

    df = df.drop(columns=['injury_status_risk', 'rolling_mean_weight_3']) # Cleanup temp column

    # --- 4. FINAL CLEANUP ---

    # Drop the temporary rolling sum columns used for the spike ratio
    df = df.drop(columns=[f'rolling_sum_snaps_{short_window}', f'rolling_sum_snaps_{long_window}'], errors='ignore')
    df = df[df['is_seed'] == False].drop(columns=['is_seed'])

    # Create binary columns for each position group
    for group, positions in position_groups.items():
        df[f'{group}'] = df['position_label'].isin(positions).astype(int)

    return df

In [10]:
import joblib
from xgboost import XGBRegressor
import os

context_forecast_df = pd.read_csv(r'C:\Users\Jack\Documents\GitHub\FFInsights\context_preds.csv')
if context_forecast_df.empty:
    raise ValueError("context_preds.csv is empty")

fatigue_model_outputs = pd.DataFrame()
prev_year = pd.DataFrame()

# Load models once outside the loop
model_paths = {
    'future_snaps': r'C:\Users\Jack\Documents\GitHub\FFInsights\fatigue_snaps_model.pkl',
    'fatigue_epa': r'C:\Users\Jack\Documents\GitHub\FFInsights\fatigue_epa_model.pkl',
    'injury_rate': r'C:\Users\Jack\Documents\GitHub\FFInsights\fatigue_injury_rate_model.pkl',
    'recovery_elasticity': r'C:\Users\Jack\Documents\GitHub\FFInsights\fatigue_recovery_elasticity_model.pkl'
}

models = {}
for key, path in model_paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model file not found: {path}")
    models[key] = joblib.load(path)

for season in range(2016, 2018):
    try:
        print(f"Processing season {season}...")
        pbp_data = nfl.import_pbp_data([season])
        depth_charts = nfl.import_depth_charts([season])
        weekly_rosters = nfl.import_weekly_rosters([season])
        injuries = nfl.import_injuries([season])

        if pbp_data.empty or depth_charts.empty or weekly_rosters.empty:
            print(f"Warning: Missing data for season {season}, skipping...")
            continue

        final_df = data_preparation(pbp_data, depth_charts, weekly_rosters, context_forecast_df, injuries, prev_year)

        if final_df.empty:
            print(f"Warning: Empty dataframe after data preparation for season {season}")
            continue

        prev_year = final_df.sort_values('week').drop_duplicates(subset=['player_id'], keep='last').copy()

        idx_cols = ['game_id', 'player_id', 'season', 'week', 'position_label', 'full_name']
        feature_cols = [col for col in final_df.columns if col not in idx_cols and pd.api.types.is_numeric_dtype(final_df[col])]

        if not feature_cols:
            print(f"Warning: No feature columns found for season {season}")
            continue

        X = final_df[feature_cols]
        
        # Check for missing values in features
        if X.isnull().any().any():
            print(f"Warning: NaN values in features for season {season}, filling with 0...")
            X = X.fillna(0)

        final_df['predicted_future_snaps'] = models['future_snaps'].predict(X)
        final_df['predicted_fatigue_epa'] = models['fatigue_epa'].predict(X)
        final_df['predicted_injury_rate'] = models['injury_rate'].predict(X)
        final_df['predicted_recovery_elasticity'] = models['recovery_elasticity'].predict(X)

        fatigue_model_outputs = pd.concat([fatigue_model_outputs, final_df[idx_cols + ['predicted_fatigue_epa', 'predicted_injury_rate', 'predicted_recovery_elasticity']]], ignore_index=True)
        print(f"Season {season} completed successfully")

    except Exception as e:
        print(f"Error processing season {season}: {str(e)}")
        continue

if fatigue_model_outputs.empty:
    raise ValueError("No data processed - fatigue_model_outputs is empty")

output_path = r'C:\Users\Jack\Documents\GitHub\FFInsights\fatigue_model_outputs.csv'
fatigue_model_outputs.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

Processing season 2016...
2016 done.
Downcasting floats.


C:\Users\Jack\AppData\Local\Temp\ipykernel_3044\1280047537.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pbp_data['ball_carrier'] = np.where(


(21981, 98)
(21981, 38)
Error processing season 2016: feature_names mismatch: ['offense_personnel_0 RB, 1 TE, 4 WR', 'qb_scramble', 'tackle_occurred', 'no_huddle', 'offense_personnel_6 OL, 1 RB, 0 TE, 3 WR', 'offense_personnel_1 RB, 3 TE, 1 WR', 'offense_personnel_2 RB, 1 TE, 2 WR', 'run_location_middle', 'offense_personnel_0 RB, 2 TE, 3 WR', 'play_type_pass', 'offense_personnel_6 OL, 2 RB, 1 TE, 1 WR', 'offense_personnel_1 RB, 0 TE, 4 WR', 'offense_personnel_2 QB, 1 RB, 1 TE, 2 WR', 'run_location_right', 'shotgun', 'offense_personnel_2 RB, 2 TE, 1 WR', 'offense_personnel_2 RB, 0 TE, 3 WR', 'offense_personnel_1 RB, 2 TE, 2 WR', 'snaps', 'run_location_left', 'rare_play_type', 'offense_personnel_6 OL, 1 RB, 2 TE, 1 WR', 'offense_personnel_1 RB, 1 TE, 3 WR', 'offense_personnel_6 OL, 1 RB, 1 TE, 2 WR', 'play_type_run', 'run_location_nan', 'offense_personnel_6 OL, 2 RB, 2 TE, 0 WR', 'yards_gained', 'was_pressure', 'time_to_throw', 'yardline_100', 'xpass', 'air_yards', 'height', 'weight', 'a

KeyboardInterrupt: 